# AKTU Autonomy Backend – Colab Runner

Run the **FastAPI backend** from **Google Colab**. No coding required — just run the cells in order.

**Run order**
1. **A. SETTINGS** — Edit the next cell if you need to change options (e.g. fresh DB, seed data, repo).
2. **B. SETUP** — Mount Drive (if used), clone repo, install deps, database setup. Optional: seed demo data and run tests.
3. **C. RUN SERVER** — Start backend and get a public URL (you will be asked for your **ngrok authtoken** once).
4. *(Optional)* Seed, run tests, or copy report metadata — see cells below.

**If Drive mount fails:** You can ignore it. The backend will use Colab’s local disk; data will be lost when the session ends, but you’ll get a working API URL for this session.


## A. SETTINGS (Edit only this)

Change the values below if needed. Then run the cell once. Do not edit the other code cells.

In [ ]:
# SETTINGS — Edit only these; then run this cell once.
USE_DRIVE = True          # True: store DB and uploads in Google Drive
RESET_DB = False          # True: wipe DB (and uploads) for a fresh run
RUN_SEED = True           # True: create demo users and sample applications during setup
RUN_TESTS = False         # True: run pytest after setup (for testing report)
PORT = 8000

REPO_URL = "https://github.com/YUVRAJ07092007/AKTU-Autonomous-Institution-Application.git"
REPO_DIR = "/content/aktu-autonomy-portal"

# For private repo: set USE_GH_PAT = True and (optionally) env AKTU_GH_PAT to your token
USE_GH_PAT = False
GH_PAT_ENV = "AKTU_GH_PAT"

## B. SETUP (mount → clone → database → optional seed/tests)

Run this cell once. It uses the SETTINGS from the cell above.

In [ ]:
from google.colab import drive
import getpass
import os
import subprocess
import sys

# Use SETTINGS from the cell above (REPO_DIR, REPO_URL, USE_DRIVE, RESET_DB, RUN_SEED, RUN_TESTS, USE_GH_PAT, GH_PAT_ENV)

# Mount Drive first (if used), then set paths
if USE_DRIVE:
    try:
        drive.mount("/content/drive")
        print("Google Drive mounted.")
    except Exception as e:
        print("Drive mount failed — using local storage for this session.")
        USE_DRIVE = False

if USE_DRIVE:
    DB_PATH = "/content/drive/MyDrive/aktu_autonomy/aktu_autonomy.db"
    UPLOAD_DIR = "/content/drive/MyDrive/aktu_autonomy/uploads"
else:
    DB_PATH = "/content/aktu_data/aktu_autonomy.db"
    UPLOAD_DIR = "/content/aktu_data/uploads"

# Optional: fresh start (wipe DB and uploads)
if RESET_DB:
    import shutil
    for path in [DB_PATH, UPLOAD_DIR]:
        try:
            if os.path.isfile(path):
                os.remove(path)
                print("Deleted:", path)
            elif os.path.isdir(path):
                shutil.rmtree(path)
                print("Removed dir:", path)
        except Exception as e:
            print("Skip", path, "—", e)
    print("Fresh start: DB cleared")

os.environ["AKTU_DB_PATH"] = DB_PATH
os.environ["UPLOAD_DIR"] = UPLOAD_DIR
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
os.makedirs(UPLOAD_DIR, exist_ok=True)

# Clone or pull (PAT only for private repo)
if not os.path.exists(REPO_DIR):
    print("Cloning repo…")
    if USE_GH_PAT or os.getenv(GH_PAT_ENV):
        token = os.getenv(GH_PAT_ENV) or getpass.getpass("GitHub PAT (repo scope): ")
        auth_url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", auth_url, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo exists; pulling latest…")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "-r", "backend/requirements.txt", "pyngrok"], check=True)
os.environ.setdefault("JWT_SECRET", "colab-secret-do-not-use-in-production")
os.environ.setdefault("ENV", "dev")

# Database setup (automatic)
BACKEND_DIR = os.path.join(REPO_DIR, "backend")
os.environ["PYTHONPATH"] = BACKEND_DIR
os.chdir(BACKEND_DIR)
result = subprocess.run(["alembic", "upgrade", "head"], capture_output=True, text=True)
if result.returncode != 0:
    err = (result.stderr or "") + (result.stdout or "")
    if "already exists" in err:
        subprocess.run(["alembic", "stamp", "head"], check=True, cwd=BACKEND_DIR)
    else:
        print(result.stdout or result.stderr)
        result.check_returncode()

# Optional: seed demo data (output appears below)
if RUN_SEED:
    subprocess.run([sys.executable, "scripts/seed_synthetic_data.py"], cwd=BACKEND_DIR, env=os.environ.copy())

# Optional: run tests
if RUN_TESTS:
    env = os.environ.copy()
    env["ENV"] = "test"
    env["JWT_SECRET"] = "test-secret"
    r = subprocess.run(["pytest", "-q", "-ra"], cwd=BACKEND_DIR, env=env)
    print("pytest exit code:", r.returncode, "(0 = all passed)")

print("\nDB path:", os.environ["AKTU_DB_PATH"])
print("Uploads:", os.environ["UPLOAD_DIR"])
print("Storage:", "Google Drive" if USE_DRIVE else "local (session only)")
print("✅ Setup complete")


## C. RUN SERVER (backend + ngrok)

When you run the next cell, an **input box** will appear — paste your **ngrok authtoken** and press Enter.  
(Get it from: https://dashboard.ngrok.com/get-started/your-authtoken)

In [ ]:
import getpass
import os
import threading
import time
import urllib.request

import uvicorn
from pyngrok import ngrok

# Use PORT from SETTINGS; REPO_DIR, BACKEND_DIR from setup
BACKEND_DIR = os.path.join(REPO_DIR, "backend")
os.chdir(BACKEND_DIR)

# Avoid multiple tunnels when re-running this cell
try:
    ngrok.kill()
except Exception:
    pass

ngrok_token = os.environ.get("NGROK_AUTHTOKEN") or getpass.getpass("Ngrok authtoken (paste below and press Enter): ")
ngrok.set_auth_token(ngrok_token)
public_url = ngrok.connect(PORT, "http").public_url

def run_backend():
    uvicorn.run("app.main:app", host="0.0.0.0", port=PORT, reload=False)

thread = threading.Thread(target=run_backend, daemon=True)
thread.start()

# Wait for backend to be healthy (up to ~10 s)
health_url = f"http://127.0.0.1:{PORT}/api/health"
for _ in range(20):
    try:
        with urllib.request.urlopen(health_url, timeout=1) as _:
            print("✅ Server healthy")
            break
    except Exception:
        time.sleep(0.5)
else:
    print("❌ Server failed to start — check errors above (port in use, missing deps, or import error).")

print("\n--- Use these URLs ---")
print("Base URL (for report):", public_url)
print("Swagger UI:", public_url + "/docs")
print("Health:", public_url + "/api/health")
print("OpenAPI JSON:", public_url + "/openapi.json")
print("\nWhat to do next: Open Swagger → Authorize (paste token if you use auth) → try GET /api/health")


In [ ]:
# (Optional) Inspect — where data is stored
import os
print("PWD:", os.getcwd())
print("DB:", os.environ.get("AKTU_DB_PATH"))
print("Uploads:", os.environ.get("UPLOAD_DIR"))
print("Storage: Google Drive" if os.environ.get("AKTU_DB_PATH", "").startswith("/content/drive") else "Storage: local (session only)")



In [ ]:
# D. SEED (optional) — Run again to add demo users and applications
# Creates 7 users, 2 institutions, 6 applications. See this cell's output for the user table.
import os
import subprocess
import sys
BACKEND_DIR = os.path.join(REPO_DIR, "backend")
env = os.environ.copy()
env["PYTHONPATH"] = BACKEND_DIR
result = subprocess.run(
    [sys.executable, "scripts/seed_synthetic_data.py"],
    cwd=BACKEND_DIR,
    env=env,
)
if result.returncode != 0:
    print("❌ Seed script exited with code:", result.returncode)
else:
    print("Seed completed — see this cell's output for created users (password: Test@123).")

In [ ]:
# E. RUN TESTS (optional) — Backend pytest for testing report
import os
import subprocess
BACKEND_DIR = os.path.join(REPO_DIR, "backend")
env = os.environ.copy()
env["ENV"] = "test"
env["JWT_SECRET"] = "test-secret"
env["PYTHONPATH"] = BACKEND_DIR
result = subprocess.run(["pytest", "-q", "-ra"], cwd=BACKEND_DIR, env=env)
print("Exit code:", result.returncode, "(0 = all passed)")

In [ ]:
# F. REPORT METADATA — Copy into your testing report
# BASE_URL = the "Base URL" printed by the RUN SERVER cell
import subprocess
commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True)
branch = subprocess.run(["git", "-C", REPO_DIR, "branch", "--show-current"], capture_output=True, text=True)
print("Git commit (for report):", (commit.stdout or "").strip())
print("Git branch (for report):", (branch.stdout or "").strip())

In [ ]:
# (Optional) Stop tunnel — closes ngrok so the public URL stops working
# To fully stop the server: Runtime → Restart runtime
try:
    from pyngrok import ngrok
    ngrok.kill()
    print("Ngrok tunnel closed.")
except Exception as e:
    print("No tunnel to close or already closed.")

## Testing and validation documentation

- [Testing and validation plan](https://github.com/YUVRAJ07092007/AKTU-Autonomous-Institution-Application/blob/main/docs/TESTING_AND_VALIDATION_PLAN.md) — plan, scenarios, report template.
- [Testing guide](https://github.com/YUVRAJ07092007/AKTU-Autonomous-Institution-Application/blob/main/docs/TESTING_GUIDE.md) — run app, seed data, manual and automated tests.
- [Testing report template](https://github.com/YUVRAJ07092007/AKTU-Autonomous-Institution-Application/blob/main/docs/TESTING_REPORT_TEMPLATE.md) — process and product report.

**Common problems**
- **/docs doesn’t open** — Use the Swagger URL from the RUN SERVER cell; allow pop-ups if blocked.
- **401 Unauthorized** — Log in via Swagger (POST /api/auth/login) and click Authorize with `Bearer <token>`.
- **403 Forbidden** — Your user role may not be allowed; use a different seeded user (see seed output).
- **ngrok fails or tunnel closed** — Re-run the RUN SERVER cell (it will create a new tunnel). Restart runtime to fully stop the server.